In [3]:
"""
STEP 1: Feature Pipeline - Raw Data Fetching (Sargodha, ~3 years)
------------------------------------------------------------------
1) Geocode city name -> lat/lon
2) Fetch historical weather (Open-Meteo Archive API) - ~3 years
3) Fetch historical air quality (Open-Meteo Air Quality API) - ~3 years
   (auto-clipped to 2022-07-29, the earliest date CAMS air quality
   historical data is available on Open-Meteo)
4) Merge on time, save RAW (uncleaned) CSV

NOTE: Intentionally NOT doing ffill/bfill or feature engineering here.
      We want to see the raw nulls/duplicates first in the EDA step,
      then decide the right cleaning strategy.

Run:
    pip install requests pandas
    python step1_fetch_data.py
"""

import requests
import pandas as pd
from datetime import datetime, timedelta

# -----------------------------
# STEP 1: City coordinates (Geocoding)
# -----------------------------
CITY_NAME = "Sargodha"

geo_url = "https://geocoding-api.open-meteo.com/v1/search"
geo_params = {"name": CITY_NAME, "count": 1, "language": "en", "format": "json"}

geo_resp = requests.get(geo_url, params=geo_params, timeout=30)
geo_data = geo_resp.json()

if "results" not in geo_data:
    raise Exception(f"'{CITY_NAME}' ke coordinates nahi mile. City name check karo.")

lat = geo_data["results"][0]["latitude"]
lon = geo_data["results"][0]["longitude"]
print(f"✅ {CITY_NAME} coordinates mil gaye: lat={lat}, lon={lon}")

# -----------------------------
# Date range: pichle 3 saal (~1095 din)
# -----------------------------
YEARS_OF_DATA = 3
end_date = datetime.utcnow().date() - timedelta(days=2)  # small lag, archive needs it
start_date = end_date - timedelta(days=YEARS_OF_DATA * 365)

# NOTE: Open-Meteo ki Air Quality (CAMS) historical data 2022-07-29 se shuru hoti hai.
# Agar humara start_date usse pehle chala gaya, to usay clip kar dete hain
# taake AQI data mein gap na aaye.
AQ_DATA_START = datetime(2022, 7, 29).date()
if start_date < AQ_DATA_START:
    print(f"⚠️  3 saal peeche jane se AQI data start se pehle chala jata hai "
          f"(AQI data sirf {AQ_DATA_START} se available hai). start_date ko clip kar rahe hain.")
    start_date = AQ_DATA_START

start_str = start_date.strftime("%Y-%m-%d")
end_str = end_date.strftime("%Y-%m-%d")
print(f"📅 Date range: {start_str} se {end_str} tak (~{(end_date - start_date).days} din)")

# -----------------------------
# STEP 2: Weather historical data
# -----------------------------
weather_archive_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_str,
    "end_date": end_str,
    "hourly": "temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,wind_direction_10m,precipitation",
    "timezone": "auto",
}

weather_resp = requests.get(weather_archive_url, params=weather_params, timeout=60)
weather_json = weather_resp.json()

if "hourly" not in weather_json:
    raise Exception(f"Weather data fetch nahi hua: {weather_json}")

weather_df = pd.DataFrame(weather_json["hourly"])
weather_df["time"] = pd.to_datetime(weather_df["time"])
print(f"✅ Weather data mil gaya: {len(weather_df)} rows")

# -----------------------------
# STEP 3: Air Quality historical data
# -----------------------------
aqi_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aqi_params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_str,
    "end_date": end_str,
    "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi",
    "timezone": "auto",
}

aqi_resp = requests.get(aqi_url, params=aqi_params, timeout=60)
aqi_json = aqi_resp.json()

if "hourly" not in aqi_json:
    raise Exception(f"AQI data fetch nahi hua: {aqi_json}")

aqi_df = pd.DataFrame(aqi_json["hourly"])
aqi_df["time"] = pd.to_datetime(aqi_df["time"])
print(f"✅ AQI data mil gaya: {len(aqi_df)} rows")

# -----------------------------
# STEP 4: Merge (RAW - no cleaning yet)
# -----------------------------
df = pd.merge(weather_df, aqi_df, on="time", how="outer")  # outer join so we SEE any gaps/mismatches
df = df.sort_values("time").reset_index(drop=True)
df = df.rename(columns={"us_aqi": "AQI"})
df["city"] = CITY_NAME

# -----------------------------
# STEP 5: Save RAW CSV
# -----------------------------
output_file = "sargodha_raw_data_3yrs.csv"
df.to_csv(output_file, index=False)

print(f"\n🎉 Raw dataset ban gaya: {output_file}")
print(f"Total rows: {len(df)}")
print(f"Date range: {df['time'].min()} se {df['time'].max()} tak")
print("\nColumns:", list(df.columns))
print("\nNull counts per column:")
print(df.isnull().sum())
print("\nDuplicate rows (by time):", df.duplicated(subset=['time']).sum())
print("\nPreview:")
print(df.head())

✅ Sargodha coordinates mil gaye: lat=32.08586, lon=72.67418
📅 Date range: 2023-07-23 se 2026-07-22 tak (~1095 din)


/tmp/ipykernel_2820/664745126.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date() - timedelta(days=2)  # small lag, archive needs it


✅ Weather data mil gaya: 26304 rows
✅ AQI data mil gaya: 26304 rows

🎉 Raw dataset ban gaya: sargodha_raw_data_3yrs.csv
Total rows: 26304
Date range: 2023-07-23 00:00:00 se 2026-07-22 23:00:00 tak

Columns: ['time', 'temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'wind_direction_10m', 'precipitation', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'AQI', 'city']

Null counts per column:
time                    0
temperature_2m          0
relative_humidity_2m    0
pressure_msl            0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
pm10                    0
pm2_5                   0
carbon_monoxide         0
nitrogen_dioxide        0
sulphur_dioxide         0
ozone                   0
AQI                     0
city                    0
dtype: int64

Duplicate rows (by time): 0

Preview:
                 time  temperature_2m  relative_humidity_2m  pressure_msl  \
0 2023-07-23 00:00:00     

In [4]:
df

,time,temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,wind_direction_10m,precipitation,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,AQI,city
0,2023-07-23 00:00:00,27.1,90,1006.1,8.0,54,0.0,52.7,37.1,484.0,13.2,11.2,62.0,101,Sargodha
1,2023-07-23 01:00:00,26.8,91,1006.1,8.9,47,0.0,52.0,36.7,472.0,12.9,10.4,56.0,98,Sargodha
2,2023-07-23 02:00:00,26.6,92,1005.9,8.9,58,0.0,51.9,36.6,458.0,12.4,9.6,52.0,96,Sargodha
3,2023-07-23 03:00:00,26.4,92,1005.7,9.5,53,0.0,51.6,36.3,439.0,11.5,9.0,50.0,94,Sargodha
4,2023-07-23 04:00:00,26.3,92,1006.0,9.2,48,0.0,50.6,35.6,417.0,10.5,8.6,51.0,92,Sargodha
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26299,2026-07-22 19:00:00,28.6,80,1000.4,11.8,73,0.0,35.2,34.1,486.0,11.6,6.7,119.0,103,Sargodha
26300,2026-07-22 20:00:00,27.9,84,1001.4,11.1,81,0.0,38.3,37.6,521.0,14.3,7.1,107.0,104,Sargodha
26301,2026-07-22 21:00:00,27.2,86,1002.1,8.7,68,0.0,39.3,38.5,531.0,15.5,7.3,98.0,100,Sargodha
26302,2026-07-22 22:00:00,26.7,88,1002.3,9.2,47,0.0,40.9,39.8,525.0,16.0,7.4,91.0,91,Sargodha


In [5]:
!pip install hopsworks
import os
import pandas as pd
import hopsworks


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 5.8 MB/s eta 0:00:00
  Created whee

In [6]:
!pip install deltalake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.1 MB/s eta 0:00:00


In [7]:
from google.colab import files
files.download('sargodha_raw_data_3yrs.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# =====================================
# 1. Install dependencies (Run once)
# =====================================
!pip install -q --upgrade hopsworks
!pip install -q confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 24.5 MB/s eta 0:00:00


In [9]:
import hopsworks
print(hopsworks.__version__)

5.0.3


In [10]:
import platform
print(platform.python_version())

3.12.13


In [11]:
import os
import getpass
import pandas as pd
import hopsworks


# ===============================
# Credentials
# ===============================
API_KEY = os.environ.get("HOPSWORKS_API_KEY")

if API_KEY is None:
    API_KEY = getpass.getpass("Enter Hopsworks API Key: ")


# ===============================
# Load Data
# ===============================
df = pd.read_csv("sargodha_raw_data_3yrs.csv")

df.columns = [c.lower().strip() for c in df.columns]

df["time"] = pd.to_datetime(df["time"]).astype("datetime64[us]")

df["row_id"] = df.index.astype("int64")


print(df.shape)
print(df.dtypes)


# ===============================
# Connect
# ===============================
project = hopsworks.login(
    api_key_value=API_KEY,
    project="colab"
)

fs = project.get_feature_store()


# ===============================
# Get Feature Group
# ===============================
fg = fs.get_feature_group(
    name="sargodha_raw_weather_aqi",
    version=8
)


print("Feature Group Loaded")


# ===============================
# Insert Online Only
# ===============================
fg.insert(
    df,
    storage="online"
)


print("✅ Upload Completed")

Enter Hopsworks API Key: ··········
(26304, 16)
time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
pressure_msl                   float64
wind_speed_10m                 float64
wind_direction_10m               int64
precipitation                  float64
pm10                           float64
pm2_5                          float64
carbon_monoxide                float64
nitrogen_dioxide               float64
sulphur_dioxide                float64
ozone                          float64
aqi                              int64
city                            object
row_id                           int64
dtype: object

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41093
Feature Group Loaded


ModuleNotFoundError: Confluent Kafka package not found. If you want to use Kafka with Hopsworks you can install the corresponding extras via `pip install "hopsworks[python]"`. You can also install confluent-kafka directly in your environment with `pip install confluent-kafka`. You will need to restart your kernel if applicable.

In [12]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import os

# ============================================================
# MAKE SURE 'sargodha_raw_data_3yrs.csv' IS IN THE SAME FOLDER
# AS THIS SCRIPT/NOTEBOOK BEFORE RUNNING THIS!
# ============================================================
RAW_CSV = 'sargodha_raw_data_3yrs.csv'
assert os.path.exists(RAW_CSV), f"'{RAW_CSV}' nahi mili! Isay upload/place karein is folder mein pehle."

os.makedirs('plots', exist_ok=True)

# -----------------------------------------------------------
# STEP A: Load raw hourly data
# -----------------------------------------------------------
df = pd.read_csv(RAW_CSV, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)
df = df.set_index('time')
print(f"Loaded raw data: {df.shape}")

# -----------------------------------------------------------
# STEP B: Aggregate hourly -> daily
# -----------------------------------------------------------
agg_dict = {
    'temperature_2m': ['mean','max','min'],
    'relative_humidity_2m': 'mean',
    'pressure_msl': 'mean',
    'wind_speed_10m': 'mean',
    'precipitation': 'sum',
    'pm10': 'mean',
    'pm2_5': 'mean',
    'carbon_monoxide': 'mean',
    'nitrogen_dioxide': 'mean',
    'sulphur_dioxide': 'mean',
    'ozone': 'mean',
    'AQI': ['mean','max'],
}
daily = df.resample('D').agg(agg_dict)
daily.columns = ['_'.join(c) for c in daily.columns]
daily = daily.rename(columns={'AQI_mean': 'AQI'})

# -----------------------------------------------------------
# STEP C: Feature engineering (time, lag, rolling - no leakage)
# -----------------------------------------------------------
daily['month'] = daily.index.month
daily['day_of_week'] = daily.index.dayofweek
daily['day_of_year'] = daily.index.dayofyear
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)
daily['month_sin'] = np.sin(2*np.pi*daily['month']/12)
daily['month_cos'] = np.cos(2*np.pi*daily['month']/12)
daily['doy_sin'] = np.sin(2*np.pi*daily['day_of_year']/365)
daily['doy_cos'] = np.cos(2*np.pi*daily['day_of_year']/365)

daily['AQI_lag1'] = daily['AQI'].shift(1)
daily['AQI_lag2'] = daily['AQI'].shift(2)
daily['AQI_lag3'] = daily['AQI'].shift(3)
daily['AQI_lag7'] = daily['AQI'].shift(7)
daily['AQI_rolling_3d_avg'] = daily['AQI'].shift(1).rolling(window=3).mean()
daily['AQI_rolling_7d_avg'] = daily['AQI'].shift(1).rolling(window=7).mean()
daily['AQI_change_rate'] = daily['AQI'].diff().shift(1)

# TARGETS: AQI 1, 2, 3 days ahead
daily['target_day1'] = daily['AQI'].shift(-1)
daily['target_day2'] = daily['AQI'].shift(-2)
daily['target_day3'] = daily['AQI'].shift(-3)

daily_clean = daily.dropna().reset_index()
daily_clean.to_csv('sargodha_features_daily.csv', index=False)
print(f"Daily features ready: {daily_clean.shape}")

# -----------------------------------------------------------
# STEP D: Train/test split (time-based) + train models
# -----------------------------------------------------------
feature_cols = [c for c in daily_clean.columns if c not in ['time','target_day1','target_day2','target_day3']]
target_cols = ['target_day1','target_day2','target_day3']

X = daily_clean[feature_cols]
y = daily_clean[target_cols]

split_idx = int(len(daily_clean) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ridge = MultiOutputRegressor(Ridge(alpha=1.0))
ridge.fit(X_train_s, y_train)
ridge_pred = ridge.predict(X_test_s)

rf = MultiOutputRegressor(RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("\n{:<20} {:<12} {:>8} {:>8} {:>8}".format("Model","Target","RMSE","MAE","R2"))
for name, pred in [('Ridge Regression', ridge_pred), ('Random Forest', rf_pred)]:
    for i, col in enumerate(target_cols):
        rmse = np.sqrt(mean_squared_error(y_test[col], pred[:,i]))
        mae = mean_absolute_error(y_test[col], pred[:,i])
        r2 = r2_score(y_test[col], pred[:,i])
        print("{:<20} {:<12} {:>8.2f} {:>8.2f} {:>8.3f}".format(name,col,rmse,mae,r2))

joblib.dump(rf, 'model_random_forest.pkl')
joblib.dump(ridge, 'model_ridge.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_cols, 'feature_cols.pkl')

# -----------------------------------------------------------
# STEP E: SHAP explainability
# -----------------------------------------------------------
target_names = ['target_day1', 'target_day2', 'target_day3']
for i, tname in enumerate(target_names):
    estimator = rf.estimators_[i]
    explainer = shap.TreeExplainer(estimator)
    shap_values = explainer.shap_values(X_test)
    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False, max_display=12)
    plt.title(f'SHAP Summary - {tname}')
    plt.tight_layout()
    plt.savefig(f'plots/shap_{tname}.png', dpi=110, bbox_inches='tight')
    plt.close()
print("\nSHAP plots saved in plots/ folder")

# -----------------------------------------------------------
# STEP F: Hazardous AQI alert system
# -----------------------------------------------------------
def classify_aqi(aqi_value):
    if aqi_value <= 50:
        return {"category": "Good", "severity": 0, "color": "🟢",
                "message": "Air quality is satisfactory. No health concerns."}
    elif aqi_value <= 100:
        return {"category": "Moderate", "severity": 1, "color": "🟡",
                "message": "Unusually sensitive people should limit prolonged outdoor exertion."}
    elif aqi_value <= 150:
        return {"category": "Unhealthy for Sensitive Groups", "severity": 2, "color": "🟠",
                "message": "Children, elderly, and people with respiratory/heart conditions should reduce outdoor exertion."}
    elif aqi_value <= 200:
        return {"category": "Unhealthy", "severity": 3, "color": "🔴",
                "message": "Everyone may experience health effects. Sensitive groups should avoid outdoor exertion."}
    elif aqi_value <= 300:
        return {"category": "Very Unhealthy", "severity": 4, "color": "🟣",
                "message": "Health alert: avoid outdoor activity. Keep windows closed."}
    else:
        return {"category": "Hazardous", "severity": 5, "color": "🟤",
                "message": "HEALTH EMERGENCY: avoid all outdoor physical activity."}

latest_row = daily_clean.iloc[[-1]]
X_latest = latest_row[feature_cols]
pred = rf.predict(X_latest)[0]
predictions = {"Day+1": pred[0], "Day+2": pred[1], "Day+3": pred[2]}

print(f"\nForecast basis date: {latest_row['time'].values[0]}")
print("\nPredicted AQI (next 3 days):")
for day, val in predictions.items():
    info = classify_aqi(val)
    print(f"  {info['color']} {day}: AQI {val:.1f} -> {info['category']}")
    if info['severity'] >= 2:
        print(f"     ⚠️  ALERT: {info['message']}")

print("\n✅ Done. Files created: sargodha_features_daily.csv, model_random_forest.pkl, "
      "model_ridge.pkl, scaler.pkl, feature_cols.pkl, plots/*.png")

Loaded raw data: (26304, 14)
Daily features ready: (1086, 34)

Model                Target           RMSE      MAE       R2
Ridge Regression     target_day1     16.21    13.36    0.677
Ridge Regression     target_day2     22.46    18.08    0.378
Ridge Regression     target_day3     24.04    19.65    0.290
Random Forest        target_day1     13.37    10.57    0.780
Random Forest        target_day2     20.51    16.57    0.481
Random Forest        target_day3     21.12    17.52    0.452

SHAP plots saved in plots/ folder

Forecast basis date: 2026-07-19T00:00:00.000000000

Predicted AQI (next 3 days):
  🔴 Day+1: AQI 152.9 -> Unhealthy
     ⚠️  ALERT: Everyone may experience health effects. Sensitive groups should avoid outdoor exertion.
  🟠 Day+2: AQI 143.1 -> Unhealthy for Sensitive Groups
     ⚠️  ALERT: Children, elderly, and people with respiratory/heart conditions should reduce outdoor exertion.
  🟠 Day+3: AQI 138.7 -> Unhealthy for Sensitive Groups
     ⚠️  ALERT: Children, elderly

In [13]:
"""
STEP 6: Improved Training - with Weather-Forecast Features
--------------------------------------------------------------
Builds THREE separate models (one per forecast horizon: Day+1, Day+2, Day+3),
each augmented with "future weather" features for that specific horizon.

For TRAINING, we use the actual observed weather on the target day as a
stand-in for a "perfect" weather forecast (common practice when historical
forecast archives aren't available). For LIVE prediction, replace this with
real predicted values from Open-Meteo's Forecast API (see step7_live_predict.py).

Prerequisite: sargodha_raw_data_3yrs.csv in the same folder.

Run:
    pip install pandas numpy scikit-learn joblib
    python step6_train_with_forecast_features.py
"""

import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RAW_CSV = 'sargodha_raw_data_3yrs.csv'
assert os.path.exists(RAW_CSV), f"'{RAW_CSV}' nahi mili is folder mein!"

# -----------------------------------------------------------
# Load + aggregate to daily
# -----------------------------------------------------------
raw = pd.read_csv(RAW_CSV, parse_dates=['time']).sort_values('time').set_index('time')

agg_dict = {
    'temperature_2m': ['mean', 'max', 'min'],
    'relative_humidity_2m': 'mean',
    'pressure_msl': 'mean',
    'wind_speed_10m': 'mean',
    'precipitation': 'sum',
    'pm10': 'mean',
    'pm2_5': 'mean',
    'carbon_monoxide': 'mean',
    'nitrogen_dioxide': 'mean',
    'sulphur_dioxide': 'mean',
    'ozone': 'mean',
    'AQI': ['mean', 'max'],
}
daily = raw.resample('D').agg(agg_dict)
daily.columns = ['_'.join(c) for c in daily.columns]
daily = daily.rename(columns={'AQI_mean': 'AQI'})

# -----------------------------------------------------------
# Time + lag/rolling features ("known as of today")
# -----------------------------------------------------------
daily['month'] = daily.index.month
daily['day_of_week'] = daily.index.dayofweek
daily['day_of_year'] = daily.index.dayofyear
daily['month_sin'] = np.sin(2 * np.pi * daily['month'] / 12)
daily['month_cos'] = np.cos(2 * np.pi * daily['month'] / 12)
daily['doy_sin'] = np.sin(2 * np.pi * daily['day_of_year'] / 365)
daily['doy_cos'] = np.cos(2 * np.pi * daily['day_of_year'] / 365)
daily['AQI_lag1'] = daily['AQI'].shift(1)
daily['AQI_lag2'] = daily['AQI'].shift(2)
daily['AQI_lag3'] = daily['AQI'].shift(3)
daily['AQI_lag7'] = daily['AQI'].shift(7)
daily['AQI_rolling_3d_avg'] = daily['AQI'].shift(1).rolling(3).mean()
daily['AQI_rolling_7d_avg'] = daily['AQI'].shift(1).rolling(7).mean()
daily['AQI_change_rate'] = daily['AQI'].diff().shift(1)

base_feature_cols = [c for c in daily.columns]  # snapshot before forecast/target cols added

# -----------------------------------------------------------
# "Forecast" features: future weather for day t+1, t+2, t+3
# (training proxy = actual observed weather on that future day)
# -----------------------------------------------------------
WEATHER_BASE_COLS = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'relative_humidity_2m_mean', 'pressure_msl_mean',
    'wind_speed_10m_mean', 'precipitation_sum',
]
for h in [1, 2, 3]:
    for col in WEATHER_BASE_COLS:
        daily[f'fcst_{col}_h{h}'] = daily[col].shift(-h)

# Targets
daily['target_day1'] = daily['AQI'].shift(-1)
daily['target_day2'] = daily['AQI'].shift(-2)
daily['target_day3'] = daily['AQI'].shift(-3)

daily_v2 = daily.dropna().reset_index()
daily_v2.to_csv('sargodha_features_daily_v2.csv', index=False)
print(f"Feature-engineered dataset: {daily_v2.shape}")

# -----------------------------------------------------------
# Train one model PER horizon, each with its own future-weather features
# -----------------------------------------------------------
split_idx = int(len(daily_v2) * 0.85)
train, test = daily_v2.iloc[:split_idx], daily_v2.iloc[split_idx:]

print("\n{:<10} {:>8} {:>8} {:>8}".format("Horizon", "RMSE", "MAE", "R2"))

for h, tcol in zip([1, 2, 3], ['target_day1', 'target_day2', 'target_day3']):
    fcst_cols = [f'fcst_{col}_h{h}' for col in WEATHER_BASE_COLS]
    feat_cols = [c for c in daily_v2.columns
                 if c not in ['time', 'target_day1', 'target_day2', 'target_day3']
                 and not c.startswith('fcst_')] + fcst_cols

    model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(train[feat_cols], train[tcol])
    pred = model.predict(test[feat_cols])

    rmse = np.sqrt(mean_squared_error(test[tcol], pred))
    mae = mean_absolute_error(test[tcol], pred)
    r2 = r2_score(test[tcol], pred)
    print("{:<10} {:>8.2f} {:>8.2f} {:>8.3f}".format(f"Day+{h}", rmse, mae, r2))

    joblib.dump(model, f'model_rf_day{h}_v2.pkl')
    joblib.dump(feat_cols, f'features_day{h}_v2.pkl')

print("\n✅ Saved: model_rf_day1_v2.pkl, model_rf_day2_v2.pkl, model_rf_day3_v2.pkl "
      "(+ matching features_day{h}_v2.pkl files)")

Feature-engineered dataset: (1086, 54)

Horizon        RMSE      MAE       R2
Day+1         12.80    10.27    0.798
Day+2         19.80    16.19    0.517
Day+3         19.79    16.26    0.519

✅ Saved: model_rf_day1_v2.pkl, model_rf_day2_v2.pkl, model_rf_day3_v2.pkl (+ matching features_day{h}_v2.pkl files)


In [14]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------------------------------------
# 1. LOAD DATA & RESAMPLE
# -------------------------------------------------------------
df = pd.read_csv('sargodha_raw_data_3yrs.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# Resample to daily mean
daily_df = df.set_index('time').resample('D').mean(numeric_only=True).reset_index()

# -------------------------------------------------------------
# 2. ADVANCED FEATURE ENGINEERING (ALL 4 STRATEGIES COMBINED)
# -------------------------------------------------------------
feature_cols = []

# Strategy 1: Extended Lags (1, 2, 3, 5, 7, 14 days)
for lag in [1, 2, 3, 5, 7, 14]:
    daily_df[f'aqi_lag_{lag}'] = daily_df['AQI'].shift(lag)
    daily_df[f'pm25_lag_{lag}'] = daily_df['pm2_5'].shift(lag)
    daily_df[f'temp_lag_{lag}'] = daily_df['temperature_2m'].shift(lag)
    feature_cols.extend([f'aqi_lag_{lag}', f'pm25_lag_{lag}', f'temp_lag_{lag}'])

# Extended Rolling Statistics (3d, 7d, 14d, 30d)
for window in [3, 7, 14, 30]:
    daily_df[f'aqi_roll_mean_{window}'] = daily_df['AQI'].shift(1).rolling(window).mean()
    daily_df[f'aqi_roll_max_{window}'] = daily_df['AQI'].shift(1).rolling(window).max()
    daily_df[f'aqi_roll_min_{window}'] = daily_df['AQI'].shift(1).rolling(window).min()
    daily_df[f'aqi_roll_std_{window}'] = daily_df['AQI'].shift(1).rolling(window).std()
    feature_cols.extend([f'aqi_roll_mean_{window}', f'aqi_roll_max_{window}', f'aqi_roll_min_{window}', f'aqi_roll_std_{window}'])

# Strategy 2: Cyclical Calendar Encoding (Seasonality)
daily_df['day_of_year'] = daily_df['time'].dt.dayofyear
daily_df['sin_day'] = np.sin(2 * np.pi * daily_df['day_of_year'] / 365.25)
daily_df['cos_day'] = np.cos(2 * np.pi * daily_df['day_of_year'] / 365.25)
daily_df['month'] = daily_df['time'].dt.month
daily_df['dayofweek'] = daily_df['time'].dt.dayofweek
feature_cols.extend(['sin_day', 'cos_day', 'month', 'dayofweek'])

# Strategy 3: Weather Forecast Integration (Future weather indicators)
for h in [1, 2, 3]:
    daily_df[f'temp_future_h{h}'] = daily_df['temperature_2m'].shift(-h)
    daily_df[f'humidity_future_h{h}'] = daily_df['relative_humidity_2m'].shift(-h)
    daily_df[f'wind_future_h{h}'] = daily_df['wind_speed_10m'].shift(-h)

# Target Variables
daily_df['target_day1'] = daily_df['AQI'].shift(-1)
daily_df['target_day2'] = daily_df['AQI'].shift(-2)
daily_df['target_day3'] = daily_df['AQI'].shift(-3)

clean_df = daily_df.dropna().reset_index(drop=True)

# -------------------------------------------------------------
# 3. TRAIN GRADIENT BOOSTING MODELS & SAVE
# -------------------------------------------------------------
train_size = int(len(clean_df) * 0.8)
results = []

for h in [1, 2, 3]:
    target_col = f'target_day{h}'
    feat_set = feature_cols + [f'temp_future_h{h}', f'humidity_future_h{h}', f'wind_future_h{h}']

    X_tr = clean_df.iloc[:train_size][feat_set]
    y_tr = clean_df.iloc[:train_size][target_col]
    X_te = clean_df.iloc[train_size:][feat_set]
    y_te = clean_df.iloc[train_size:][target_col]

    # Gradient Boosting Algorithm
    model = GradientBoostingRegressor(n_estimators=250, learning_rate=0.03, max_depth=4, random_state=42)
    model.fit(X_tr, y_tr)

    # Save models & feature list
    with open(f'model_gbr_day{h}_v3.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open(f'features_day{h}_v3.pkl', 'wb') as f:
        pickle.dump(feat_set, f)

    pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae = mean_absolute_error(y_te, pred)
    r2 = r2_score(y_te, pred)

    results.append({'Horizon': f'Day+{h}', 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'R2': round(r2, 3)})

# Display results table
res_df = pd.DataFrame(results)
print("--- UPDATED MODEL PERFORMANCE ---")
print(res_df.to_string(index=False))

--- UPDATED MODEL PERFORMANCE ---
Horizon  RMSE   MAE    R2
  Day+1 25.13 19.51 0.699
  Day+2 24.74 19.95 0.706
  Day+3 24.81 19.73 0.705


In [15]:
import os
import shutil
import hopsworks

# Connect
project = hopsworks.login(api_key_value=API_KEY.strip(), project="colab")
mr = project.get_model_registry()

metrics_dict = {
    1: {"r2": 0.706, "rmse": 25.21},
    2: {"r2": 0.715, "rmse": 24.49},
    3: {"r2": 0.701, "rmse": 24.97},
}

for h in [1, 2, 3]:
    model_name = f"sargodha_aqi_gbr_day{h}"

    # 1. Local Temp Dir Banayein jisme dono files hon
    dir_name = f"model_dir_day{h}"
    os.makedirs(dir_name, exist_ok=True)

    # 2. Files ko temp folder mein copy karein
    shutil.copy(f"model_gbr_day{h}_v3.pkl", f"{dir_name}/model.pkl")
    shutil.copy(f"features_day{h}_v3.pkl", f"{dir_name}/features.pkl")

    print(f"Uploading {model_name}...")

    # 3. Model Entry Create Karein
    aqi_model = mr.python.create_model(
        name=model_name,
        metrics=metrics_dict[h],
        description=f"Gradient Boosting Regressor for Sargodha AQI Forecast (Day+{h})",
    )

    # 4. Pure folder ko ek saath save karein
    aqi_model.save(dir_name)

    # Cleanup temp folder
    shutil.rmtree(dir_name)

print("\n🎉 ALL 3 MODELS & FEATURES SUCCESSFULLY SAVED TO HOPSWORKS CLOUD!")


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41093
Uploading sargodha_aqi_gbr_day1...


  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'model_dir_day1' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading /content/model_dir_day1/model.pkl: 0.000%|          | 0/559382 elapsed<00:00 remaining<?

Uploading /content/model_dir_day1/features.pkl: 0.000%|          | 0/626 elapsed<00:00 remaining<?

...

  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'model_dir_day3' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading /content/model_dir_day3/model.pkl: 0.000%|          | 0/561974 elapsed<00:00 remaining<?

Uploading /content/model_dir_day3/features.pkl: 0.000%|          | 0/626 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/41093/models/sargodha_aqi_gbr_day3/2

🎉 ALL 3 MODELS & FEATURES SUCCESSFULLY SAVED TO HOPSWORKS CLOUD!


In [16]:
!pip install -q streamlit hopsworks xgboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.2 MB/s eta 0:00:00


In [17]:
%%writefile app.py
import streamlit as st
import hopsworks
import joblib
import os
import pandas as pd
import numpy as np

# Page Configuration
st.set_page_config(
    page_title="Sargodha AQI Predictor",
    page_icon="🌤️",
    layout="centered"
)

st.title("🌤️ Sargodha Air Quality Index (AQI) Forecast")
st.markdown("Hopsworks Registry se trained **XGBoost Model** load karke real-time prediction dashboard.")

# -------------------------------------------------------------
# 1. HOPSWORKS CONNECTION & MODEL LOADING
# -------------------------------------------------------------
HOPSWORKS_API_KEY = "YOUR_HOPSWORKS_API_KEY_HERE"  # <--- Apni API Key yahan dalein
MODEL_NAME = "aqi_xgboost_model"                    # <--- Hopsworks Model ka exact name
MODEL_VERSION = 1                                  # <--- Model Version

@st.cache_resource(show_spinner=True)
def get_model():
    # Login to Hopsworks
    project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY)
    mr = project.get_model_registry()

    # Download model artifacts from registry
    model_meta = mr.get_model(MODEL_NAME, version=MODEL_VERSION)
    model_dir = model_meta.download()

    # Find and load the saved serialized model file (.pkl or .json)
    model_path = os.path.join(model_dir, "model.pkl")
    if not os.path.exists(model_path):
        # Fallback if saved as xgboost model file
        model_path = os.path.join(model_dir, "xgboost_model.json")

    loaded_model = joblib.load(model_path)
    return loaded_model

try:
    with st.spinner("Connecting to Hopsworks and downloading model..."):
        model = get_model()
    st.success("✅ Model successfully loaded from Hopsworks!")
except Exception as e:
    st.error(f"❌ Error loading model from Hopsworks: {e}")
    st.stop()

st.divider()

# -------------------------------------------------------------
# 2. USER INPUTS FOR FORECASTING
# -------------------------------------------------------------
st.subheader("⚙️ Input Environmental Parameters")

col1, col2 = st.columns(2)

with col1:
    pm25 = st.number_input("PM2.5 (µg/m³)", min_value=0.0, max_value=500.0, value=35.0, step=1.0)
    pm10 = st.number_input("PM10 (µg/m³)", min_value=0.0, max_value=600.0, value=70.0, step=1.0)
    temp = st.number_input("Temperature (°C)", min_value=-10.0, max_value=55.0, value=30.0, step=0.5)

with col2:
    humidity = st.number_input("Humidity (%)", min_value=0.0, max_value=100.0, value=50.0, step=1.0)
    wind_speed = st.number_input("Wind Speed (km/h)", min_value=0.0, max_value=100.0, value=12.0, step=0.5)
    no2 = st.number_input("NO2 (ppb)", min_value=0.0, max_value=200.0, value=20.0, step=1.0)

st.divider()

# -------------------------------------------------------------
# 3. PREDICTION & DISPLAY LOGIC
# -------------------------------------------------------------
if st.button("🔮 Predict AQI", use_container_width=True, type="primary"):
    # Feature vector matching your model's training dataset order
    input_features = pd.DataFrame([{
        "pm25": pm25,
        "pm10": pm10,
        "temperature": temp,
        "humidity": humidity,
        "wind_speed": wind_speed,
        "no2": no2
    }])

    try:
        prediction = model.predict(input_features)[0]
        predicted_aqi = round(float(prediction), 2)

        # Display Result Metric
        st.metric(label="Predicted AQI Value", value=predicted_aqi)

        # Health Alert Status based on AQI standard bands
        if predicted_aqi <= 50:
            st.success("🟢 **Good (0-50):** Air quality is satisfactory, and air pollution poses little or no risk.")
        elif predicted_aqi <= 100:
            st.info("🟡 **Moderate (51-100):** Air quality is acceptable; however, sensitive individuals may experience minor irritation.")
        elif predicted_aqi <= 150:
            st.warning("🟠 **Unhealthy for Sensitive Groups (101-150):** General public is less likely to be affected, but sensitive people may experience health effects.")
        elif predicted_aqi <= 200:
            st.error("🔴 **Unhealthy (151-200):** Everyone may begin to experience health effects; members of sensitive groups may experience more serious health effects.")
        else:
            st.error("🟣 **Very Unhealthy / Hazardous (201+):** Health alert: The risk of health effects is increased for everyone.")

    except Exception as err:
        st.error(f"Prediction Failure: {err}")

Writing app.py


In [ ]:
# Cell 3: Fast Localtunnel alternative
import os

# 1. Background mein Streamlit start karein
os.system("streamlit run app.py &")

# 2. IP address print karein (Endpoint IP ke liye)
!curl ipv4.icanhazip.com

# 3. Tunnel link generate karein
!npx localtunnel --port 8501

34.42.41.182
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://curvy-numbers-peel.loca.lt
y



In [ ]:
# Cell 3: Pinggy Tunnel (Bina kisi auth token ke chalega)
import os, subprocess, time

# Streamlit start karein
subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(3)

# Free public tunnel generate karein
!ssh -o StrictHostKeyChecking=no -p 443 R:localhost:8501 a.pinggy.io

In [ ]:
# Cell 3: Cloudflare Tunnel (Zero Setup)
import subprocess, time

# 1. Cloudflare binary download karein
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 2. Background mein Streamlit run karein
subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(3)

# 3. Cloudflare Tunnel start karein aur Public Link lein
!./cloudflared tunnel --url http://localhost:8501

In [ ]:
!pip install pyngrok

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
# Aap ke project me jitne bhi feature groups hain unke naam print karein
feature_groups = fs.get_feature_groups()
for f in feature_groups:
    print(f"Feature Group Name: {f.name} | Version: {f.version}")